In [ ]:
import os
import requests
import pandas as pd

# 1. Thiết lập đường dẫn lưu file
SAVE_PATH = "../data/01_raw/step1_pm25.csv"

# 2. Cấu hình chung
LAT, LON = 10.7769, 106.7009
START_DATE = "2024-01-01"
END_DATE = "2026-04-01"
TIMEZONE = "Asia/Bangkok"

# 3. Gọi API Air Quality
url_aq = "https://air-quality-api.open-meteo.com/v1/air-quality"
params_aq = {
    "latitude": LAT, 
    "longitude": LON,
    "hourly": "pm2_5",
    "start_date": START_DATE, 
    "end_date": END_DATE,
    "timezone": TIMEZONE
}

try:
    response = requests.get(url_aq, params=params_aq)
    response.raise_for_status() # Kiểm tra lỗi HTTP (ví dụ: 404, 500)
    res_aq = response.json()
    
    # 4. Chuyển thành DataFrame
    df_pm25 = pd.DataFrame({
        "datetime": pd.to_datetime(res_aq['hourly']['time']),
        "pm25": res_aq['hourly']['pm2_5']
    })

    # 5. Kiểm tra Missing Data
    print("-" * 30)
    print("Kiểm tra dữ liệu PM2.5:")
    missing_pm25 = df_pm25.isnull().sum()
    print(missing_pm25)

    if missing_pm25['pm25'] > 0:
        print(f"Có {missing_pm25['pm25']} giờ bị thiếu dữ liệu PM2.5")
    else:
        print("Không có giá trị Null.")
    print("-" * 30)

    # 6. Kiểm tra thư mục và lưu file
    os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)
    df_pm25.to_csv(SAVE_PATH, index=False)
    
    print(f"Đã tải xong {len(df_pm25)} dòng PM2.5.")
    print(f"File đã được lưu tại: {SAVE_PATH}")

except requests.exceptions.RequestException as e:
    print(f"Lỗi khi kết nối API: {e}")

# Hiển thị 5 dòng đầu tiên
df_pm25.head()

------------------------------
Kiểm tra dữ liệu PM2.5:
datetime    0
pm25        0
dtype: int64
Không có giá trị Null.
------------------------------
Đã tải xong 19728 dòng PM2.5.
File đã được lưu tại: ../data/01_raw/step1_pm25.csv


,datetime,pm25
0,2024-01-01 00:00:00,17.6
1,2024-01-01 01:00:00,17.2
2,2024-01-01 02:00:00,18.8
3,2024-01-01 03:00:00,21.8
4,2024-01-01 04:00:00,24.4


In [2]:
# 1. Thiết lập đường dẫn lưu file thời tiết
SAVE_PATH_WEATHER = "../data/01_raw/step2_weather.csv"

# 2. Gọi API Weather Archive
url_weather = "https://archive-api.open-meteo.com/v1/archive"
params_weather = {
    "latitude": LAT, 
    "longitude": LON,
    "hourly": "temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,wind_direction_10m",
    "start_date": START_DATE, 
    "end_date": END_DATE,
    "timezone": TIMEZONE
}

try:
    response_w = requests.get(url_weather, params=params_weather)
    response_w.raise_for_status() # Kiểm tra lỗi HTTP (ví dụ: 404, 500)
    res_w = response_w.json()
    
    # 3. Chuyển thành DataFrame
    df_weather = pd.DataFrame({
        "datetime": pd.to_datetime(res_w['hourly']['time']),
        "temp": res_w['hourly']['temperature_2m'],
        "humidity": res_w['hourly']['relative_humidity_2m'],
        "precip": res_w['hourly']['precipitation'],
        "wind_speed": res_w['hourly']['wind_speed_10m'],
        "wind_dir": res_w['hourly']['wind_direction_10m']
    })

    # 4. Kiểm tra Missing Data
    print("-" * 30)
    print("Kiểm tra dữ liệu thời tiết:")
    missing_weather = df_weather.isnull().sum()
    print(missing_weather)

    if missing_weather.sum() > 0:
        print("Có dữ liệu bị thiếu trong các cột thời tiết")
    else:
        print("Không có giá trị Null.")
    print("-" * 30)

    # 5. Kiểm tra thư mục và lưu file
    os.makedirs(os.path.dirname(SAVE_PATH_WEATHER), exist_ok=True)
    df_weather.to_csv(SAVE_PATH_WEATHER, index=False)
    
    print(f"Đã tải xong {len(df_weather)} dòng thời tiết.")
    print(f"File đã được lưu tại: {SAVE_PATH_WEATHER}")

except requests.exceptions.RequestException as e:
    print(f"Lỗi khi kết nối API: {e}")

# Hiển thị 5 dòng đầu tiên trên Jupyter Notebook
df_weather.head()

------------------------------
Kiểm tra dữ liệu thời tiết:
datetime      0
temp          0
humidity      0
precip        0
wind_speed    0
wind_dir      0
dtype: int64
Không có giá trị Null.
------------------------------
Đã tải xong 19728 dòng thời tiết.
File đã được lưu tại: ../data/01_raw/step2_weather.csv


,datetime,temp,humidity,precip,wind_speed,wind_dir
0,2024-01-01 00:00:00,25.6,88,0.0,5.7,145
1,2024-01-01 01:00:00,25.4,90,0.0,3.6,143
2,2024-01-01 02:00:00,25.3,88,0.0,0.5,135
3,2024-01-01 03:00:00,25.0,91,0.0,4.7,9
4,2024-01-01 04:00:00,24.6,92,0.0,8.0,352


In [3]:
# 1. Thiết lập đường dẫn đọc và lưu file
PATH_PM25 = "../data/01_raw/step1_pm25.csv"
PATH_WEATHER = "../data/01_raw/step2_weather.csv"
SAVE_PATH_MASTER = "../data/02_interim/pm25_weather.csv"

try:
    # Đọc lại dữ liệu từ 2 file đã lưu (để đảm bảo tính độc lập)
    df1 = pd.read_csv(PATH_PM25)
    df2 = pd.read_csv(PATH_WEATHER)

    # Chuyển cột datetime về định dạng chuẩn của Pandas
    df1['datetime'] = pd.to_datetime(df1['datetime'])
    df2['datetime'] = pd.to_datetime(df2['datetime'])

    # 2. Tiến hành Merge (Inner Join)
    # Dùng inner join để đảm bảo giờ nào có đủ cả PM2.5 và Thời tiết thì mới giữ lại
    df_master = pd.merge(df1, df2, on="datetime", how="inner")

    # 3. Kiểm tra dữ liệu thiếu (Missing values) sau khi gộp
    print("-" * 30)
    print("Kiểm tra dữ liệu sau khi gộp:")
    print(f"Tổng số dòng dữ liệu hiện có: {len(df_master)}")
    
    missing_data = df_master.isnull().sum()
    print("\nSố giá trị thiếu trong từng cột:")
    print(missing_data)
    print("-" * 30)

    # 4. Kiểm tra thư mục và lưu file Master cuối cùng
    os.makedirs(os.path.dirname(SAVE_PATH_MASTER), exist_ok=True)
    df_master.to_csv(SAVE_PATH_MASTER, index=False)
    
    print(f"Đã gộp xong, File đã được lưu tại: {SAVE_PATH_MASTER}")

except FileNotFoundError as e:
    print(f"Không tìm thấy file dữ liệu đầu vào: {e}")
except Exception as e:
    print(f"Đã xảy ra lỗi ngoài ý muốn: {e}")

# Hiển thị 5 dòng đầu tiên để kiểm tra cấu trúc bảng cuối cùng
df_master.head()

------------------------------
Kiểm tra dữ liệu sau khi gộp:
Tổng số dòng dữ liệu hiện có: 19728

Số giá trị thiếu trong từng cột:
datetime      0
pm25          0
temp          0
humidity      0
precip        0
wind_speed    0
wind_dir      0
dtype: int64
------------------------------
Đã gộp xong, File đã được lưu tại: ../data/02_interim/pm25_weather.csv


,datetime,pm25,temp,humidity,precip,wind_speed,wind_dir
0,2024-01-01 00:00:00,17.6,25.6,88,0.0,5.7,145
1,2024-01-01 01:00:00,17.2,25.4,90,0.0,3.6,143
2,2024-01-01 02:00:00,18.8,25.3,88,0.0,0.5,135
3,2024-01-01 03:00:00,21.8,25.0,91,0.0,4.7,9
4,2024-01-01 04:00:00,24.4,24.6,92,0.0,8.0,352
